# 🎨 AI Product Image Generator
### Kaggle Competition — End-to-End Pipeline

**Pipeline:**
1. LLM-powered prompt engineering (Qwen2.5-0.5B-Instruct)
2. Image generation (Stable Diffusion v1.5)
3. Gradio interactive UI
4. Batch processing of 15 product descriptions

**Optimized for:** Google Colab T4 GPU | float16 | Memory-efficient

---
## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q \
    torch torchvision \
    transformers accelerate \
    diffusers \
    xformers \
    gradio \
    Pillow \
    pandas \
    scipy \
    ftfy

print("✅ All dependencies installed.")

---
## Cell 2 — Imports & Configuration

In [ ]:
import os
import gc
import time
import random
import warnings
warnings.filterwarnings("ignore")

import torch
import pandas as pd
from PIL import Image
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

# ── Configuration ──────────────────────────────────────────────
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

LLM_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"

NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5
IMAGE_SIZE = 512

OUTPUT_DIR = Path("outputs")
IMAGE_DIR = OUTPUT_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
torch.manual_seed(SEED)
random.seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print(f"🖥️  Device : {DEVICE}")
print(f"📐 Dtype  : {DTYPE}")
if DEVICE == "cuda":
    print(f"🎮 GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print("✅ Configuration ready.")

---
## Cell 3 — Load LLM (Prompt Engineer)

In [ ]:
print(f"📦 Loading LLM: {LLM_MODEL_ID} ...")

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_ID,
    trust_remote_code=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
llm_model.eval()

print(f"✅ LLM loaded — {sum(p.numel() for p in llm_model.parameters()) / 1e6:.0f}M params")

---
## Cell 4 — Load Stable Diffusion Pipeline

In [ ]:
print(f"📦 Loading Stable Diffusion: {SD_MODEL_ID} ...")

sd_pipe = StableDiffusionPipeline.from_pretrained(
    SD_MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)

# Use faster scheduler
sd_pipe.scheduler = DPMSolverMultistepScheduler.from_config(sd_pipe.scheduler.config)

sd_pipe = sd_pipe.to(DEVICE)

# Memory optimizations
sd_pipe.enable_attention_slicing()
try:
    sd_pipe.enable_xformers_memory_efficient_attention()
    print("  ✓ xformers enabled")
except Exception:
    print("  ⚠ xformers not available — using attention slicing only")

print("✅ Stable Diffusion pipeline ready.")

---
## Cell 5 — Prompt Engineering Function

In [ ]:
SYSTEM_PROMPT = """You are a professional product photographer and prompt engineer specializing in e-commerce imagery.
Your task: convert a simple product description into a concise, high-quality Stable Diffusion prompt.

Rules:
- Output ONLY the prompt, no explanations or extra text.
- Keep the prompt under 60 words.
- Always include: product type, material/texture, studio lighting, clean background, camera angle, and quality keywords.
- Always include these CLIP-aligned keywords: "product photography", "studio lighting", "white background", "sharp focus", "ultra realistic", "8k".
- Use a consistent structure: [product] [details] [lighting] [background] [camera] [quality].
- Do NOT include negative prompts, special tokens, or markdown formatting."""


def generate_prompt(description: str) -> str:
    """Convert a plain product description into a professional SD prompt using the LLM."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Product: {description}"},
    ]

    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    prompt = llm_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Clean up: remove quotes, markdown artifacts
    prompt = prompt.strip('"').strip("'").strip()
    for prefix in ["Prompt:", "prompt:", "Output:", "output:"]:
        if prompt.startswith(prefix):
            prompt = prompt[len(prefix):].strip()

    # Truncate to ~65 words max for consistency
    words = prompt.split()
    if len(words) > 65:
        prompt = " ".join(words[:65])

    # Ensure core CLIP keywords are present
    clip_keywords = ["product photography", "studio lighting", "sharp focus"]
    for kw in clip_keywords:
        if kw.lower() not in prompt.lower():
            prompt += f", {kw}"

    return prompt


# Quick test
test_prompt = generate_prompt("red leather handbag")
print(f"📝 Test prompt ({len(test_prompt.split())} words):")
print(test_prompt)

---
## Cell 6 — Image Generation Function

In [ ]:
NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, ugly, noisy, text, watermark, "
    "oversaturated, cartoon, illustration, painting, sketch, bad proportions, "
    "cropped, out of frame, duplicate, morbid, mutilated"
)


def generate_image(prompt: str, seed: int = SEED) -> Image.Image:
    """Generate a product image from a text prompt using Stable Diffusion."""

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
        result = sd_pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            width=IMAGE_SIZE,
            height=IMAGE_SIZE,
            generator=generator,
        )

    return result.images[0]


# Quick test
print("🖼️  Generating test image...")
test_image = generate_image(test_prompt)
test_image.save(IMAGE_DIR / "_test.png")
print("✅ Test image saved.")
display(test_image)

---
## Cell 7 — Full Pipeline Function

In [ ]:
def full_pipeline(description: str) -> tuple:
    """End-to-end: description → prompt → image."""
    prompt = generate_prompt(description)
    image = generate_image(prompt)
    return prompt, image


print("✅ Pipeline function defined.")

---
## Cell 8 — Gradio UI

In [ ]:
import gradio as gr


def gradio_generate(description: str):
    """Wrapper for Gradio interface."""
    if not description or not description.strip():
        return "Please enter a product description.", None
    prompt, image = full_pipeline(description.strip())
    return prompt, image


with gr.Blocks(
    title="AI Product Image Generator",
    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"),
) as demo:

    gr.Markdown(
        """# 🎨 AI Product Image Generator
        Enter a simple product description and get a professional product photo.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_text = gr.Textbox(
                label="Product Description",
                placeholder="e.g. red leather handbag with gold buckle",
                lines=3,
            )
            generate_btn = gr.Button("🚀 Generate", variant="primary", size="lg")

        with gr.Column(scale=1):
            output_prompt = gr.Textbox(label="Engineered Prompt", lines=4, interactive=False)

    output_image = gr.Image(label="Generated Product Image", type="pil", height=512)

    generate_btn.click(
        fn=gradio_generate,
        inputs=input_text,
        outputs=[output_prompt, output_image],
    )

    gr.Examples(
        examples=[
            ["ceramic coffee mug with floral pattern"],
            ["wireless Bluetooth headphones in black"],
            ["luxury gold wristwatch"],
        ],
        inputs=input_text,
    )

# Launch inline (Colab-compatible)
demo.launch(inline=True, share=False, debug=False, quiet=True)

---
## Cell 9 — Product Descriptions Dataset

In [ ]:
# ── 15 Product Descriptions ──────────────────────────────────
product_descriptions = [
    "red leather handbag with gold buckle",
    "wireless Bluetooth headphones in black",
    "luxury gold wristwatch with leather strap",
    "ceramic coffee mug with floral pattern",
    "stainless steel water bottle, matte black",
    "running shoes with neon green accents",
    "vintage wooden sunglasses",
    "minimalist white sneakers",
    "premium fountain pen, silver and black",
    "organic cotton tote bag, natural beige",
    "smart fitness tracker, rose gold",
    "handmade scented candle in glass jar",
    "leather wallet, dark brown, bifold",
    "portable Bluetooth speaker, navy blue",
    "silk necktie with paisley pattern",
]

print(f"📋 {len(product_descriptions)} product descriptions loaded.")
for i, desc in enumerate(product_descriptions, 1):
    print(f"  {i:2d}. {desc}")

---
## Cell 10 — Batch Processing

In [ ]:
results = []

print("="*70)
print("🚀 BATCH PROCESSING — 15 PRODUCTS")
print("="*70)

total_start = time.time()

for idx, description in enumerate(product_descriptions):
    item_num = idx + 1
    print(f"\n{'─'*60}")
    print(f"[{item_num:2d}/15] {description}")
    print(f"{'─'*60}")

    start = time.time()

    # Generate prompt
    prompt = generate_prompt(description)
    print(f"  📝 Prompt ({len(prompt.split())} words): {prompt[:120]}...")

    # Generate image with unique seed per product
    image_seed = SEED + idx
    image = generate_image(prompt, seed=image_seed)

    # Save image
    image_filename = f"product_{item_num:02d}.png"
    image_path = IMAGE_DIR / image_filename
    image.save(image_path, "PNG")

    elapsed = time.time() - start
    print(f"  ✅ Saved: {image_path} ({elapsed:.1f}s)")

    results.append({
        "id": item_num,
        "description": description,
        "generated_prompt": prompt,
        "image_path": str(image_path),
        "seed": image_seed,
        "time_seconds": round(elapsed, 1),
    })

    # Free CUDA cache periodically
    if DEVICE == "cuda" and idx % 5 == 4:
        torch.cuda.empty_cache()
        gc.collect()

total_elapsed = time.time() - total_start

print(f"\n{'='*70}")
print(f"✅ BATCH COMPLETE — {len(results)} images generated in {total_elapsed:.1f}s")
print(f"   Average: {total_elapsed / len(results):.1f}s per image")
print(f"{'='*70}")

---
## Cell 11 — Save Results to CSV

In [ ]:
# Save to CSV
df = pd.DataFrame(results)
csv_path = OUTPUT_DIR / "prompts.csv"
df.to_csv(csv_path, index=False)

print(f"📄 Results saved to: {csv_path}")
print(f"\n{df.to_string(index=False)}")

---
## Cell 12 — Display Generated Images Gallery

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 5, figsize=(25, 16))
fig.suptitle("AI Product Image Generator — Results", fontsize=20, fontweight="bold", y=0.98)

for idx, row in df.iterrows():
    ax = axes[idx // 5][idx % 5]
    img = Image.open(row["image_path"])
    ax.imshow(img)
    ax.set_title(row["description"], fontsize=9, fontweight="bold", pad=6)
    ax.axis("off")

plt.tight_layout(rect=[0, 0, 1, 0.96])
gallery_path = OUTPUT_DIR / "gallery.png"
plt.savefig(gallery_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

print(f"🖼️  Gallery saved to: {gallery_path}")

---
## Cell 13 — Kaggle Evaluation Compatibility

This cell provides variables and file paths in the format expected by standard Kaggle evaluation cells.

In [ ]:
# ── Variables for Kaggle Evaluation ──────────────────────────

# List of generated prompts (in order)
generated_prompts = [r["generated_prompt"] for r in results]

# List of image file paths (in order)
generated_images = [r["image_path"] for r in results]

# List of product descriptions (in order)
descriptions = [r["description"] for r in results]

# DataFrame with all results
submission_df = df.copy()

# Verify all images exist
for img_path in generated_images:
    assert os.path.exists(img_path), f"Missing: {img_path}"

print(f"✅ Evaluation variables ready.")
print(f"   generated_prompts : list[str]  — {len(generated_prompts)} items")
print(f"   generated_images  : list[str]  — {len(generated_images)} items")
print(f"   descriptions      : list[str]  — {len(descriptions)} items")
print(f"   submission_df     : DataFrame  — {submission_df.shape}")
print(f"   prompts.csv       : {csv_path}")

---
## Cell 14 — CLIP Score Evaluation (Optional Self-Check)

In [ ]:
%%capture install_clip
!pip install -q open_clip_torch

In [ ]:
import open_clip
from PIL import Image as PILImage
import numpy as np

print("📊 Computing CLIP scores for self-evaluation...")

# Load CLIP model (lightweight ViT-B/32)
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

clip_scores = []

for idx, row in df.iterrows():
    img = PILImage.open(row["image_path"]).convert("RGB")
    img_tensor = clip_preprocess(img).unsqueeze(0).to(DEVICE)
    text_tokens = clip_tokenizer([row["generated_prompt"]]).to(DEVICE)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)

        img_features /= img_features.norm(dim=-1, keepdim=True)
        txt_features /= txt_features.norm(dim=-1, keepdim=True)

        similarity = (img_features @ txt_features.T).item()

    clip_scores.append(similarity)
    print(f"  [{idx+1:2d}/15] CLIP: {similarity:.4f} | {row['description']}")

avg_clip = np.mean(clip_scores)
print(f"\n📊 Average CLIP Score: {avg_clip:.4f}")
print(f"   Min: {min(clip_scores):.4f}  |  Max: {max(clip_scores):.4f}")

# Cleanup CLIP model to free memory
del clip_model, clip_preprocess, clip_tokenizer
torch.cuda.empty_cache()
gc.collect()

print("✅ CLIP evaluation complete. Model unloaded to free memory.")

---
## Cell 15 — Cosine Similarity of Prompts (Optional Self-Check)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("📊 Computing prompt cosine similarity...")

vectorizer = TfidfVectorizer()

# Compare each generated prompt against its original description
similarities = []
for idx, row in df.iterrows():
    tfidf = vectorizer.fit_transform([row["description"], row["generated_prompt"]])
    sim = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    similarities.append(sim)
    print(f"  [{idx+1:2d}/15] Similarity: {sim:.4f} | {row['description']}")

avg_sim = np.mean(similarities)
print(f"\n📊 Average Cosine Similarity: {avg_sim:.4f}")
print(f"   Min: {min(similarities):.4f}  |  Max: {max(similarities):.4f}")
print("✅ Prompt similarity evaluation complete.")

---
## Cell 16 — Final Summary

In [ ]:
print("\n" + "="*70)
print("  🎉 AI PRODUCT IMAGE GENERATOR — COMPLETE")
print("="*70)
print(f"""\n  📋 Products processed  : {len(results)}
  📝 Prompts generated   : {len(generated_prompts)}
  🖼️  Images generated    : {len(generated_images)}
  📄 CSV saved           : {csv_path}
  📁 Images folder       : {IMAGE_DIR}
  🖼️  Gallery             : {gallery_path}
  ⏱️  Total time          : {total_elapsed:.1f}s
  ⚡ Avg per image       : {total_elapsed / len(results):.1f}s
""")

if DEVICE == "cuda":
    mem_used = torch.cuda.max_memory_allocated() / 1e9
    print(f"  💾 Peak GPU memory    : {mem_used:.2f} GB")

print("  ✅ All outputs are Kaggle-evaluation compatible.")
print("="*70)